# Fixed-beta spread 的 ADF：固定样本均值与 rolling mean 为什么不是一回事

本课回答一个很容易混淆的问题：`quant_dashboard.py` 标题中的 ADF p-value 究竟在检验什么？固定 beta 页面能否照搬？又为什么不应把 rolling mean 当作 ADF 的检验对象？

结论先说：ADF 应检验**固定定义的 spread 水平序列** $S_t=Y_t-\beta X_t$，在一个预先确定的样本窗内是否有单位根。减去一个固定常数（例如该窗的 spread mean）不会改变带截距 ADF 的结论；rolling mean 是随时间变化的滤波器，改变了序列的随机过程，不能用原来的 p-value 含义解释。

## 1. 原 dashboard 的统计对象

旧版 dashboard 先在固定的 training window 拟合 $\alpha,\beta$，随后对同一个训练窗的残差做 ADF：

$$S_t=Y_t-(\alpha+\beta X_t),\qquad \text{ADF}(S_1,\ldots,S_T).$$

因此它的 p-value 回答的是：在 ADF 设定、滞后阶数与该训练样本下，能否拒绝 $S_t$ 有单位根的原假设。它**不**是当前 z-score 的概率，也不是对未来必然均值回复的保证。

fixed-beta 版本应保持 beta 冻结：$S_t=Y_t-\beta X_t$。若需要截距，只是在报告或交易基线中减去一个常数；不要在同一页面悄悄重新拟合 beta。

## 2. ADF 的数学本质

最常见的带截距 ADF 回归写为：

$$\Delta S_t=c+\gamma S_{t-1}+\sum_{i=1}^{p}\phi_i\Delta S_{t-i}+\varepsilon_t.$$

检验 $H_0:\gamma=0$（单位根）对 $H_1:\gamma<0$（均值回复/平稳）。p-value 来自 $\gamma$ 的 t 统计量的 Dickey--Fuller 非标准分布。`autolag` 只是在候选滞后阶数中选择 $p$；它不是 rolling window。

若 spread 是 AR(1)，$S_t-\mu=\rho(S_{t-1}-\mu)+u_t$，且 $|\rho|<1$，则 $\gamma=\rho-1<0$；若 $\rho=1$，便是随机游走，无法由 ADF 拒绝单位根。

## 3. 固定 spread mean 为什么可以

令 $C$ 为一个不随 $t$ 改变的数，可以是训练窗均值 $\bar S$、事先设定的基线或截距。定义 $R_t=S_t-C$。则

$$\Delta R_t=\Delta S_t,\qquad R_{t-1}=S_{t-1}-C.$$

代入 ADF 回归后，$-\gamma C$ 只会并入截距 $c$。所以当 ADF 包含常数项（`statsmodels.adfuller(..., regression='c')` 的默认设定）时，$S_t$ 与 $S_t-C$ 的 ADF t 值和 p-value 在数值上应相同（同一滞后选择与数值精度下）。

这里的“固定 window / 固定 mean”指：先确定一个样本区间，在该区间内 $C$ 是一个常数。它仍有样本内估计的不确定性，但没有把序列从一个随机过程变成另一个随机过程。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

rng = np.random.default_rng(7)
n = 1_500
rho = 0.94
eps = rng.normal(scale=1.0, size=n)
stationary_spread = np.empty(n)
stationary_spread[0] = 20.0
for t in range(1, n):
    stationary_spread[t] = 20.0 + rho * (stationary_spread[t - 1] - 20.0) + eps[t]

fixed_mean = stationary_spread.mean()
def adf_summary(values, label):
    stat, pvalue, used_lag, nobs, critical, _ = adfuller(values, regression='c', autolag='AIC')
    return pd.Series({'series': label, 'ADF statistic': stat, 'p-value': pvalue, 'lags': used_lag, 'observations': nobs})

pd.DataFrame([
    adf_summary(stationary_spread, 'S_t'),
    adf_summary(stationary_spread - fixed_mean, 'S_t - fixed sample mean'),
]).set_index('series')


上面两行的 p-value 应相同或只差浮点误差。这说明 ADF 的对象不是“距离哪一条水平线多远”，而是序列是否具有单位根。均值仅决定平稳过程围绕哪个常数摆动。

## 4. rolling mean 改变了什么

rolling mean 不是常数，而是

$$M_t=\frac1w\sum_{j=0}^{w-1}S_{t-j}. $$

它每一根 bar 都改变，并且相邻 $M_t$ 共享 $w-1$ 个观测。若检验 $M_t$，你检验的是低通滤波后的序列；若检验偏离量 $D_t=S_t-M_t$，你检验的是高通滤波后的序列。两者都不再是原始 spread 的 ADF。

最关键的反例：若 $S_t=S_{t-1}+u_t$ 是随机游走，则

$$S_t-M_t=\frac1w\sum_{j=0}^{w-2}(w-1-j)\,u_{t-j},$$

这是有限阶 MA 过程，天然平稳。也就是说，对随机游走先减 rolling mean 再 ADF，常会错误地得到很小的 p-value；这不是发现了协整，而是滤波器机械地消去了单位根。

In [ ]:
# 一个完全没有均值回复的随机游走
random_walk = 100 + np.cumsum(rng.normal(size=n))
w = 60
rolling_mean = pd.Series(random_walk).rolling(w, min_periods=w).mean()
rolling_deviation = (pd.Series(random_walk) - rolling_mean).dropna().to_numpy()

pd.DataFrame([
    adf_summary(random_walk, 'random walk S_t'),
    adf_summary(rolling_mean.dropna(), 'rolling mean M_t'),
    adf_summary(rolling_deviation, 'S_t - rolling mean M_t'),
]).set_index('series')


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
view = slice(0, 350)
ax.plot(random_walk[view], color='#94a3b8', lw=1.2, label='random walk $S_t$')
ax.plot(rolling_mean.iloc[view], color='#2563eb', lw=2, label=f'{w}-bar rolling mean $M_t$')
ax.set_title('rolling mean 是随时间移动的滤波器，不是固定基线')
ax.set_xlabel('bar')
ax.legend()
ax.grid(alpha=.2)
plt.show()


## 5. 在 fixed-beta dashboard 中应如何报告

推荐的、可解释的字段是：

1. `ADF p-value (selected range)`：用户在全局导航图选定的区间，以当前冻结 beta 算 $S_t=Y_t-\beta X_t$，对原始 spread 做一次带截距 ADF。
2. 可选的 `ADF p-value (training range)`：只对研究时预先锁定的训练窗计算，并与 out-of-sample 表现分开解读。
3. `Rolling mean`、`Rolling sigma` 和 z-score：作为逐 bar 的交易状态量，允许随时间更新；它们不是 ADF 的输入。

若不断改变导航区间、反复挑选最小 p-value，再据此宣布平稳，会产生 data snooping（多重检验）偏差。正式配置应先锁定训练窗、冻结 beta、记录 ADF，再在后续样本检验交易规则。

In [ ]:
# 可直接移入 fixed-beta 页面的一次性计算函数。
# 输入必须是当前 beta 下、选定区间内的原始 spread；不要传 spread_mean 或 z-score。
def fixed_beta_adf(spread: pd.Series) -> dict:
    values = pd.Series(spread).replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
    if len(values) < 30 or np.isclose(np.std(values), 0.0):
        return {'adf_stat': np.nan, 'p_value': np.nan, 'used_lag': np.nan, 'nobs': len(values)}
    stat, p_value, used_lag, nobs, _, _ = adfuller(values, regression='c', autolag='AIC')
    return {'adf_stat': stat, 'p_value': p_value, 'used_lag': used_lag, 'nobs': nobs}

fixed_beta_adf(stationary_spread)
